# RedLine — Kaggle Ollama Gateway (host model cho web chạy local)

Notebook này **chỉ host model**: Ollama + gateway API (chọn/tải/xóa model, tiến trình tải, OpenAI `/v1`) qua ngrok.
Web RedLine chạy trên máy bạn bằng Docker và kết nối tới đây. Nếu muốn chạy cả web trên Kaggle, dùng `kaggle_redline.ipynb`.

Kaggle Settings: bật **Internet**, chọn **GPU T4 x2**. Trong **Add-ons → Secrets**, tạo và Attach:
- `NGROK_AUTH_TOKEN` (bắt buộc)
- `GATEWAY_TOKEN` (khuyến nghị, ≥ 16 ký tự). Nếu không có, notebook sinh token ngẫu nhiên và in ra **một lần** để bạn dán vào web.

Model nhẹ đề xuất (đã kiểm tra trên registry Ollama): `qwen3.5:4b` (3.4 GB), `qwen3:4b` (2.5 GB), `llama3.2:3b` (2.0 GB), `qwen3.5:9b` (6.6 GB), `llama3.1:8b` (4.9 GB), `gpt-oss:20b` (13.8 GB).

In [ ]:
from pathlib import Path
import os, sys, json, time, secrets, subprocess, shutil, urllib.request

REPO_URL = "https://github.com/TranQuangMinh-2005/RedLine.git"
BRANCH = "main"
PRELOAD_MODELS = "qwen3.5:4b"   # phân tách bằng dấu phẩy; "" = không tải trước, chọn từ web
CONTEXT_LENGTH = 8192
GATEWAY_PORT = 8080
OLLAMA_VERSION = ""              # Rỗng: bộ cài hiện hành; điền version để tái lập
REPO_DIR = Path("/kaggle/working/RedLine")
RUNTIME_DIR = REPO_DIR / "runs/kaggle"

if not Path("/kaggle/working").is_dir():
    raise RuntimeError("Notebook này dành cho Kaggle.")
if "gateway" in globals() and gateway.poll() is None:
    raise RuntimeError("Gateway đang chạy. Chạy cell Dừng trước khi cấu hình lại.")

from kaggle_secrets import UserSecretsClient
secrets_client = UserSecretsClient()
def read_secret(*names):
    for name in names:
        value = os.environ.get(name, "").strip()
        if value:
            return value
        try:
            value = secrets_client.get_secret(name).strip()
            if value:
                return value
        except Exception:
            pass
    return ""

ngrok_token = read_secret("NGROK_AUTH_TOKEN", "NGROK_AUTHTOKEN")
if not ngrok_token:
    raise RuntimeError("Thiếu NGROK_AUTH_TOKEN. Tạo Secret và bật Attach trước khi chạy.")
gateway_token = read_secret("GATEWAY_TOKEN")
generated_token = len(gateway_token) < 16
if generated_token:
    gateway_token = secrets.token_urlsafe(24)
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)
print("Cấu hình và Secrets: OK")

## 1. Lấy repo và cài môi trường Python

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO_DIR, text=True)
    if dirty.strip():
        raise RuntimeError("Repo có thay đổi chưa commit; lưu/xử lý chúng trước khi update.")
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR, check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "uv"], check=True)
subprocess.run([sys.executable, "-m", "uv", "python", "install", "3.11"], check=True)
subprocess.run([sys.executable, "-m", "uv", "sync", "--locked", "--no-dev", "--python", "3.11"],
               cwd=REPO_DIR, check=True)
PYTHON = REPO_DIR / ".venv/bin/python"
subprocess.run([str(PYTHON), "--version"], check=True)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip())

## 2. Cài Ollama

In [ ]:
import hashlib
import platform

RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
os.environ["PATH"] = "/usr/local/bin:/usr/bin:" + os.environ.get("PATH", "")
if not shutil.which("ollama") or OLLAMA_VERSION:
    sudo = [] if os.geteuid() == 0 else ["sudo"]
    if not shutil.which("zstd"):
        subprocess.run(sudo + ["apt-get", "update", "-qq"], check=True)
        subprocess.run(sudo + ["apt-get", "install", "-y", "-qq", "zstd"], check=True)
    installer = RUNTIME_DIR / "install-ollama.sh"
    urllib.request.urlretrieve("https://ollama.com/install.sh", installer)
    install_env = os.environ.copy()
    if OLLAMA_VERSION:
        install_env["OLLAMA_VERSION"] = OLLAMA_VERSION
    subprocess.run(["sh", str(installer)], env=install_env, check=True)

print("Ollama:", subprocess.check_output(["ollama", "--version"], text=True).strip())

## 3. Khởi động gateway

Launcher chạy `ollama serve`, tải trước `PRELOAD_MODELS`, chạy gateway và mở tunnel ngrok. Khi báo `ready`, cell kết thúc nhưng dịch vụ tiếp tục chạy nền.

In [ ]:
if "gateway" in globals() and gateway.poll() is None:
    raise RuntimeError("Gateway đang chạy. Dừng ở cell cuối trước khi chạy lại.")
launch_env = os.environ.copy()
launch_env.update({"NGROK_AUTHTOKEN": ngrok_token, "GATEWAY_TOKEN": gateway_token, "PYTHONUNBUFFERED": "1"})
command = [str(PYTHON), "scripts/kaggle_gateway.py", "--models", PRELOAD_MODELS,
           "--context-length", str(CONTEXT_LENGTH), "--port", str(GATEWAY_PORT),
           "--runtime-dir", str(RUNTIME_DIR)]
status_path = RUNTIME_DIR / "gateway-status.json"
status_path.unlink(missing_ok=True)
log_path = RUNTIME_DIR / "gateway-launcher.log"
with log_path.open("w") as log:
    gateway = subprocess.Popen(command, cwd=REPO_DIR, env=launch_env,
                               stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
try:
    deadline = time.monotonic() + 3600
    offset = 0
    while time.monotonic() < deadline:
        with log_path.open() as log:
            log.seek(offset)
            output = log.read()
            offset = log.tell()
        if output:
            print(output, end="", flush=True)
        if gateway.poll() is not None:
            raise RuntimeError(f"Launcher đã thoát ({gateway.returncode}). Xem log phía trên.")
        if status_path.exists():
            status = json.loads(status_path.read_text())
            if status.get("state") == "ready":
                print("\n=== Dán vào web RedLine → Model → Endpoint khác ===")
                print("Base URL:", status["openai_base_url"])
                if generated_token:
                    print("API key / token:", gateway_token, "(sinh ngẫu nhiên — không chia sẻ output notebook)")
                else:
                    print("API key / token: giá trị Secret GATEWAY_TOKEN")
                break
            if status.get("state") in {"failed", "stopped"}:
                raise RuntimeError(f"Startup {status['state']}; kiểm tra log.")
        time.sleep(2)
    else:
        raise TimeoutError("Quá 60 phút chờ khởi động")
except BaseException:
    if gateway.poll() is None:
        gateway.terminate()
        try:
            gateway.wait(timeout=90)
        except subprocess.TimeoutExpired:
            print("Cleanup còn chạy; dùng Stop session nếu cần.")
    raise

## 4. Kiểm tra API gateway (nội bộ)

Từ web: chọn **Endpoint khác**, dán Base URL + token, bấm **Kết nối**. Tab tải model hiển thị catalog, tiến trình tải và model đã cài.

In [ ]:
def gw(path, method="GET", payload=None):
    request = urllib.request.Request(
        status["local_url"] + path, method=method,
        data=None if payload is None else json.dumps(payload).encode(),
        headers={"Authorization": f"Bearer {gateway_token}", "Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.load(response)

info = gw("/gateway/info")
print("Ollama", info["ollama_version"], "| disk free", info["disk_free_gb"], "GB | GPU", [g["name"] for g in info["gpus"]])
for row in gw("/gateway/catalog")["models"]:
    print(f"{'✔' if row['installed'] else ' '} {row['id']:<14} {row['size_gb']:>5} GB  {row['note']}")
print("OpenAI /v1/models:", [m["id"] for m in gw("/v1/models")["data"]])

In [ ]:
# (Tùy chọn) Tải thêm model từ notebook và theo dõi tiến trình — web dùng đúng API này.
MODEL_TO_PULL = ""  # ví dụ "llama3.2:3b"
if MODEL_TO_PULL:
    job = gw("/gateway/pulls", "POST", {"model": MODEL_TO_PULL})
    while job["state"] == "running":
        time.sleep(2)
        job = gw(f"/gateway/pulls/{job['id']}")
        print(f"\r{job['status'][:30]:<30} {job['percent']:5.1f}%  {job['speed_bps']/1e6:6.1f} MB/s", end="")
    print("\n", job["state"], job.get("error") or "")

In [ ]:
for filename in ("gateway-launcher.log", "ollama.log", "gateway.log"):
    path = RUNTIME_DIR / filename
    if path.exists():
        print(f"\n--- {filename} (30 dòng cuối) ---")
        print("\n".join(path.read_text(errors="replace").splitlines()[-30:]))

## 5. Dừng gateway

Model cache trong `runs/kaggle/models` vẫn giữ trong phiên. Dùng **Stop session** để ngừng GPU.

In [ ]:
if "gateway" in globals() and gateway.poll() is None:
    gateway.terminate()
    try:
        gateway.wait(timeout=90)
        print("Đã dừng gateway, tunnel và Ollama do launcher tạo.")
    except subprocess.TimeoutExpired:
        print("Cleanup chưa hoàn tất. Dùng Stop session.")
else:
    print("Không có gateway đang chạy trong kernel này.")